# Data acquisition

This notebook builds the dataset and gathers every structure the analysis uses. Candidate proteins are
taken from public tables (SIFTS, the wwPDB method and resolution indexes, and UniProt) and filtered by a
fixed set of rules, after which one chain is chosen for each experimental method. The experimental
structures are downloaded from the RCSB PDB and the AlphaFold2 models from the AlphaFold Database; the
OpenFold3 models are predicted through the NVIDIA NIM service, with a multiple sequence alignment for
every protein. Files already on disk are not downloaded or predicted again.

**Input:** network access and an NVIDIA NIM API key, set as `NIM_API_KEY=nvapi-…` in a `.env` file at the
repository root (keys are issued at https://build.nvidia.com/openfold/openfold3).  
**Output:** `proteins.csv`, the structure files in `data/` and `results/dataset_selection_counts.csv`.

## Setup

In [13]:
import sys, os, json
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")

from src.downloader import (
    download_rcsb,
    download_alphafold2,
    fetch_uniprot_sequence,
    fetch_msa,
    predict_openfold3,
)
from src.parser import load_structure, get_ca_coords_and_seq
from src.metadata import extract_metadata
import src.dataset as ds

load_dotenv()

nim_key_loaded = bool(os.environ.get("NIM_API_KEY"))
print(f"NIM_API_KEY found: {nim_key_loaded}")
if not nim_key_loaded:
    print("  Add NIM_API_KEY=nvapi-... to a .env file at the repo root.")

NIM_API_KEY found: True


In [14]:
DATA_DIR = "data"
EXP_DIR = os.path.join(DATA_DIR, "experimental")
AF2_DIR = os.path.join(DATA_DIR, "alphafold2")
OF3_DIR = os.path.join(DATA_DIR, "openfold3")

for d in [EXP_DIR, AF2_DIR, OF3_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"  {d}")

proteins = pd.read_csv("proteins.csv", dtype=str, keep_default_na=False)
print(f"\n{len(proteins)} proteins currently in manifest:")
display(proteins[["protein_name", "uniprot_id", "pdb_xray", "pdb_nmr", "pdb_cryoem"]])

  data/experimental
  data/alphafold2
  data/openfold3

151 proteins currently in manifest:


,protein_name,uniprot_id,pdb_xray,pdb_nmr,pdb_cryoem
0,Type IV pilin PilE1,A0A0B7GU52,,6I2O,8PFB
1,Toxin co-regulated pilus biosynthesis protein ...,A0AAI7ZGD2,,2L4W,6GYB
2,ATP synthase epsilon chain (B1MLW3),B1MLW3,,7VIL,9VKR
3,Type II secretion system core protein G,E0SM38,,29JD,11ME
4,"Acyl carrier protein, mitochondrial",O14561,6ODD,2DNW,7OF0
...,...,...,...,...,...
146,Dynein axonemal light chain 1,Q9XHH2,5YXM,1DS9,8GLV
147,Cofilin-2 (Q9Y281),Q9Y281,,7M0G,9Y9P
148,Ribosome maturation protein SBDS,Q9Y3A5,,2KDO,5AN9
149,Peptidyl-prolyl cis-trans isomerase-like 1,Q9Y3C6,2X7K,1XWN,9S2E


---
## Dataset selection

`proteins.csv` is generated here, not edited by hand. The rules, in order:

| # | Rule | Why |
|---|------|-----|
| 0 | Quality limits: X-ray ≤ `MAX_XRAY` Å, cryo-EM ≤ `MAX_EM` Å, NMR all | poorly determined structures are not used as references |
| 1 | The protein has **NMR and cryo-EM** structures | the comparison needs several methods per protein; NMR is the scarcest method (counts below), so the set is built outward from it, and cryo-EM is the second method of the topic. X-ray is added where it exists |
| 2 | One mature chain in UniProt (no polyprotein) | experiment and prediction must describe the same molecule |
| 3 | Mature chain length within `LENGTH` | enough residues for the metrics; the size range NMR can solve; prediction memory grows with N² and time with N³ |
| 4 | An NMR chain **and** a cryo-EM chain each model ≥ `MIN_COVERAGE` of the mature chain | the metrics summarise whole chains, so residues missing on one side would be measured as structural difference (notebook 03 measures how much) |
| 5 | An AlphaFold DB model exists | needed for the comparison |

Then **one chain per method** is chosen: best resolution for X-ray and cryo-EM, best coverage for NMR,
ties broken alphabetically. A multi-chain entry contributes one chain, so every comparison is one
chain against one chain (notebook 03 uses the other copies in the same entry as the tightest reference).

If fewer than 15 proteins remain, relax **one** rule at a time, in this order, and report which:
`MAX_EM` 4.0 → 4.5, `MIN_COVERAGE` 0.8 → 0.7, `LENGTH` upper 300 → 400.

In [15]:
MAX_XRAY, MAX_EM = 2.5, 4.0  # Å
LENGTH = (50, 300)  # mature chain, residues
MIN_COVERAGE = 0.8  # fraction of the mature chain modelled in the chain
REF_DIR = os.path.join(DATA_DIR, "reference")

ref = ds.download_reference_files(REF_DIR)
print("SIFTS version:", ds.sifts_version(ref["uniprot_segments_observed.csv.gz"]))

segments = ds.load_segments(ref["uniprot_segments_observed.csv.gz"])
chains = ds.chain_table(
    segments,
    ds.load_methods(ref["pdb_entry_type.txt"]),
    ds.load_resolution(ref["resolu.idx"]),
    MAX_XRAY,
    MAX_EM,
)

# How many proteins have structures of each method (within the quality limits)?  NMR is the scarcest.
print(chains.groupby("method")["uniprot"].nunique().to_string())
candidates = ds.proteins_with(chains, ("NMR", "Cryo-EM"))
print(
    f"NMR and cryo-EM: {len(candidates)} proteins | "
    f"NMR, cryo-EM and X-ray: {len(ds.proteins_with(chains, ('NMR', 'Cryo-EM', 'X-ray')))}"
)

  [skip] uniprot_segments_observed.csv.gz already present (force=True to refresh)
  [skip] pdb_entry_type.txt already present (force=True to refresh)
  [skip] resolu.idx already present (force=True to refresh)
SIFTS version: 2026/09/27 - 11:11
method
Cryo-EM    24250
NMR         6910
X-ray      42359
NMR and cryo-EM: 1055 proteins | NMR, cryo-EM and X-ray: 684


### Protein facts, coverage and filters

UniProt gives the mature chain of each candidate (e.g. barnase: residues 48–157 of 157).
Coverage is measured against it. The counts table below is the dataset table of the thesis.

In [16]:
facts = ds.fetch_uniprot_facts(
    candidates, cache_path=os.path.join(REF_DIR, "uniprot_facts.csv")
)
cov = ds.add_coverage(chains[chains["uniprot"].isin(candidates)], segments, facts)
both = lambda d: d[d["uniprot"].isin(ds.proteins_with(d))]

kept, counts = ds.filter_with_counts(
    cov,
    [
        ("single mature chain (no polyprotein)", lambda d: d[d["n_chains"] <= 1]),
        (
            f"mature length {LENGTH[0]}-{LENGTH[1]}",
            lambda d: d[d["mature_length"].between(*LENGTH)],
        ),
        (
            f"NMR and cryo-EM chains covering >= {MIN_COVERAGE:.0%}",
            lambda d: both(d[d["coverage"] >= MIN_COVERAGE]),
        ),
    ],
    start_label="NMR and cryo-EM, found in UniProt",
)

survivors = sorted(kept["uniprot"].unique())
af2 = {acc: ds.has_alphafold_model(acc) for acc in survivors}
kept, more = ds.filter_with_counts(
    kept, [("AlphaFold DB model exists", lambda d: d[d["uniprot"].map(af2)])]
)
counts = pd.concat([counts, more.iloc[1:]], ignore_index=True)
os.makedirs("results", exist_ok=True)
counts.to_csv("results/dataset_selection_counts.csv", index=False)
display(counts)

,step,proteins
0,"NMR and cryo-EM, found in UniProt",1055
1,single mature chain (no polyprotein),938
2,mature length 50-300,392
3,NMR and cryo-EM chains covering >= 80%,161
4,AlphaFold DB model exists,151


### One chain per method → `proteins.csv`

In [17]:
representatives = ds.pick_representatives(kept)
manifest = ds.write_manifest(representatives, facts, path="proteins.csv")
print(f"proteins.csv written: {len(manifest)} proteins")
display(manifest)

proteins = pd.read_csv("proteins.csv", dtype=str, keep_default_na=False)

proteins.csv written: 151 proteins


,protein_name,uniprot_id,organism,mature_start,mature_end,pdb_xray,chain_xray,pdb_nmr,chain_nmr,pdb_cryoem,chain_cryoem,cov_xray,cov_nmr,cov_cryoem,res_xray,res_cryoem,notes
0,Type IV pilin PilE1,A0A0B7GU52,Streptococcus sanguinis,19,157,,,6I2O,A,8PFB,C,,0.813,0.942,,3.67,
1,Toxin co-regulated pilus biosynthesis protein ...,A0AAI7ZGD2,Xanthomonas citri pv. citri (strain 306),22,139,,,2L4W,A,6GYB,A,,0.983,0.941,,3.28,
2,ATP synthase epsilon chain (B1MLW3),B1MLW3,Mycobacteroides abscessus (strain ATCC 19977 /...,1,121,,,7VIL,A,9VKR,H,,0.992,1.000,,2.79,
3,Type II secretion system core protein G,E0SM38,Dickeya dadantii (strain 3937) (Erwinia chrysa...,8,153,,,29JD,A,11ME,B,,0.836,0.904,,3.60,
4,"Acyl carrier protein, mitochondrial",O14561,Homo sapiens (Human),69,156,6ODD,A,2DNW,A,7OF0,w,0.966,0.977,0.898,2.0,2.20,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,Dynein axonemal light chain 1,Q9XHH2,Chlamydomonas reinhardtii (Chlamydomonas smithii),1,198,5YXM,A,1DS9,A,8GLV,AA,0.98,1.000,0.985,1.545,3.10,
147,Cofilin-2 (Q9Y281),Q9Y281,Homo sapiens (Human),2,166,,,7M0G,A,9Y9P,a,,1.000,1.000,,2.06,
148,Ribosome maturation protein SBDS,Q9Y3A5,Homo sapiens (Human),2,250,,,2KDO,A,5AN9,J,,1.000,1.000,,3.30,
149,Peptidyl-prolyl cis-trans isomerase-like 1,Q9Y3C6,Homo sapiens (Human),1,166,2X7K,A,1XWN,A,9S2E,S,0.988,1.000,0.958,1.15,2.40,


---
## Download experimental structures

Downloads the X-ray, NMR and cryo-EM entries listed in `proteins.csv` into `data/experimental/`.
Methods without an entry are skipped; files that already exist are not re-downloaded.


In [18]:
proteins = pd.read_csv("proteins.csv", dtype=str, keep_default_na=False)

download_log = []

for _, row in proteins.iterrows():
    name = row["protein_name"]
    uid = row["uniprot_id"]

    for method, col in [
        ("X-ray", "pdb_xray"),
        ("NMR", "pdb_nmr"),
        ("Cryo-EM", "pdb_cryoem"),
    ]:
        pdb_id = row.get(col)
        if pd.isna(pdb_id) or not str(pdb_id).strip():
            continue  # no structure for this method

        pdb_id = str(pdb_id).strip().upper()
        dest = os.path.join(EXP_DIR, f"{pdb_id}.cif")

        if os.path.exists(dest):
            continue  # already on disk

        print(f"  [{method:8s}] {name} — downloading {pdb_id}")
        try:
            download_rcsb(pdb_id, EXP_DIR)
            download_log.append(
                {"protein": name, "method": method, "pdb_id": pdb_id, "status": "ok"}
            )
        except Exception as exc:
            print(f"  [ERROR] {pdb_id}: {exc}")
            download_log.append(
                {
                    "protein": name,
                    "method": method,
                    "pdb_id": pdb_id,
                    "status": str(exc),
                }
            )

if download_log:
    print(f"\nDownload complete — {len(download_log)} file(s) downloaded or attempted.")
    display(pd.DataFrame(download_log))
else:
    print("All experimental CIF files already present — nothing to download.")

All experimental CIF files already present — nothing to download.


---
## AlphaFold2 predictions

Downloads AF2 predictions from the AlphaFold Protein Structure Database
(https://alphafold.ebi.ac.uk/) for **every protein currently in `proteins.csv`**.
The API always returns the latest model version. Existing files are skipped.


In [19]:
proteins = pd.read_csv("proteins.csv", dtype=str, keep_default_na=False)
print(f"Downloading AF2 for {len(proteins)} proteins...")

for _, row in proteins.iterrows():
    print(f"  {row['protein_name']} ({row['uniprot_id']})")
    try:
        download_alphafold2(row["uniprot_id"], AF2_DIR)
    except Exception as exc:
        print(f"    [ERROR] {exc}")

  Type IV pilin PilE1 (A0A0B7GU52)
  [skip] AF2_A0A0B7GU52.cif already exists
  Toxin co-regulated pilus biosynthesis protein Q C-terminal domain-containing protein (A0AAI7ZGD2)
  [skip] AF2_A0AAI7ZGD2.cif already exists
  ATP synthase epsilon chain (B1MLW3) (B1MLW3)
  [skip] AF2_B1MLW3.cif already exists
  Type II secretion system core protein G (E0SM38)
  [skip] AF2_E0SM38.cif already exists
  Acyl carrier protein, mitochondrial (O14561)
  [skip] AF2_O14561.cif already exists
  C-X-C motif chemokine 11 (O14625)
  [skip] AF2_O14625.cif already exists
  Ubiquitin/ISG15-conjugating enzyme E2 L6 (O14933)
  [skip] AF2_O14933.cif already exists
  Baculoviral IAP repeat-containing protein 5 (O15392)
  [skip] AF2_O15392.cif already exists
  NADH dehydrogenase [ubiquinone] 1 alpha subcomplex subunit 2 (O43678)
  [skip] AF2_O43678.cif already exists
  Dynein light chain Tctex1 (O64980)
  [skip] AF2_O64980.cif already exists
  Gamma-aminobutyric acid receptor-associated protein (O95166)
  [skip

---
## OpenFold3 predictions (with MSA)

**Why an MSA.** OpenFold3, like AlphaFold2, infers a fold mainly from evolution: the multiple
sequence alignment (MSA) of homologous sequences. With the query sequence alone (an MSA of depth 1)
the predictions of a test on the proteins of the set had a median pLDDT of 29 and none reached 70 —
not usable predictions — so every protein is predicted with an MSA.

**What is run, per protein:**
1. the UniProt canonical sequence — the sequence the AlphaFold DB model was predicted from;
2. its MSA from the NVIDIA MSA Search NIM (ColabFold MMseqs2 search of UniRef30 and the ColabFold
   environmental database), cached in `data/msas/<UniProt>/`;
3. OpenFold3 on NVIDIA NIM with that MSA and 5 diffusion samples. The top-ranked sample (highest
   `confidence_score`) becomes `data/openfold3/OF3_<UniProt>.cif`; all samples go to
   `data/openfold3/samples/`; scores and MSA depth to `OF3_<UniProt>_provenance.json`.

AF2 and OF3 models both cover the whole UniProt sequence.

Model: OpenFold3-preview (OpenFold Consortium), https://github.com/aqlaboratory/openfold-3 —
an open reproduction of AlphaFold3 (Abramson et al., *Nature* 2024, doi:10.1038/s41586-024-07487-w).
**Requires `NIM_API_KEY`** in `.env`. Existing MSAs and models are never requested again.

In [20]:
# 1. Sequences and checks. No NIM request is sent in this cell.
proteins = pd.read_csv("proteins.csv", dtype=str, keep_default_na=False)
STANDARD = set("ACDEFGHIKLMNPQRSTVWY")
FLAGS = ["standard letters", "AF2 = UniProt sequence"]

sequences, checks = {}, []
for _, row in proteins.iterrows():
    uid = row["uniprot_id"]
    check = {"uniprot_id": uid}
    try:
        seq = sequences[uid] = fetch_uniprot_sequence(uid)
        af2 = load_structure(os.path.join(AF2_DIR, f"AF2_{uid}.cif"))
        _, af2_seq = get_ca_coords_and_seq(af2, 0, "A")
        check.update(
            {
                "length": len(seq),
                "standard letters": set(seq) <= STANDARD,
                "AF2 = UniProt sequence": af2_seq == seq,
            }
        )
    except Exception as exc:  # e.g. a missing AF2 file or a UniProt time-out
        check["error"] = str(exc)
    checks.append(check)

checks = pd.DataFrame(checks).reindex(columns=["uniprot_id", "length", *FLAGS, "error"])
passed = checks[FLAGS].eq(True).all(axis=1)
print(f"{passed.sum()} of {len(checks)} proteins pass every check")
print(f"{checks['length'].sum():.0f} residues in total")
print(checks[FLAGS].eq(True).sum().to_string())  # proteins passing each check
display(checks[~passed])  # proteins failing a check, with the error if any

151 of 151 proteins pass every check
20163 residues in total
standard letters          151
AF2 = UniProt sequence    151


,uniprot_id,length,standard letters,AF2 = UniProt sequence,error


### Pilot: three proteins first

Decision rule, written before running: **go** if, for every pilot protein, the OF3 pLDDT is ≥ 70
and at most 10 points below AF2's. Otherwise look at that protein's MSA depth
and stop before the batch.

In [21]:
# 2. Pilot: thioredoxin (P0AA25), the protein with the most residues before its mature chain
#    (signal peptide / propeptide) and the longest protein.
MSA_DIR = os.path.join(DATA_DIR, "msas")
with_prepeptide = proteins.loc[
    proteins["mature_start"].astype(int).idxmax(), "uniprot_id"
]
longest = checks.loc[checks["length"].idxmax(), "uniprot_id"]
PILOT = list(dict.fromkeys(["P0AA25", with_prepeptide, longest]))

for uid in PILOT:
    if uid not in sequences:  # its UniProt request failed in cell 1
        sequences[uid] = fetch_uniprot_sequence(uid)
    msa = fetch_msa(uid, sequences[uid], save_dir=MSA_DIR)
    predict_openfold3(uid, sequences[uid], OF3_DIR, msa=msa)

  [skip] MSA for P0AA25 already exists
  [skip] OF3_P0AA25.cif already exists
  [skip] MSA for P12301 already exists
  [skip] OF3_P12301.cif already exists
  [skip] MSA for P21796 already exists
  [skip] OF3_P21796.cif already exists


In [22]:
def of3_report(uids) -> pd.DataFrame:
    """
    Per protein: whether the OF3 model has the UniProt sequence; MSA depth;
    mean pLDDT of OF3 and AF2.
    """

    def plddt(path):
        if not os.path.exists(path):
            return None
        return extract_metadata(path).plddt_mean

    rows = []
    for uid in uids:
        of3 = os.path.join(OF3_DIR, f"OF3_{uid}.cif")
        if not os.path.exists(of3):
            continue
        with open(of3.replace(".cif", "_provenance.json")) as f:
            prov = json.load(f)
        _, of3_seq = get_ca_coords_and_seq(load_structure(of3), 0, "A")
        rows.append(
            {
                "uniprot_id": uid,
                "OF3 = UniProt sequence": of3_seq == sequences.get(uid),
                "MSA depth": " + ".join(
                    str(n) for n in prov["msa_n_sequences"].values()
                ),
                "confidence": prov["sample_scores"][prov["selected_sample"]].get(
                    "confidence_score"
                ),
                "pLDDT OF3": plddt(of3),
                "pLDDT AF2": plddt(os.path.join(AF2_DIR, f"AF2_{uid}.cif")),
            }
        )
    return pd.DataFrame(rows).round(1)


of3_report(PILOT)

,uniprot_id,OF3 = UniProt sequence,MSA depth,confidence,pLDDT OF3,pLDDT AF2
0,P0AA25,True,101 + 101,0.7,86.9,95.2
1,P12301,True,103 + 101,0.2,70.6,71.6
2,P21796,True,101 + 101,0.5,81.3,93.1


### All proteins (only after the pilot passes)

Re-running the cell continues where it stopped: MSAs and models already on disk are skipped.

In [23]:
# 3. Batch
failed = {}
for i, uid in enumerate(proteins["uniprot_id"], start=1):
    print(f"[{i}/{len(proteins)}] {uid}")
    try:
        if uid not in sequences:  # its UniProt request failed in cell 1
            sequences[uid] = fetch_uniprot_sequence(uid)
        msa = fetch_msa(uid, sequences[uid], save_dir=MSA_DIR)
        predict_openfold3(uid, sequences[uid], OF3_DIR, msa=msa)
    except Exception as exc:
        failed[uid] = str(exc)
        print(f"  [ERROR] {exc}")

print(f"\n{len(proteins) - len(failed)} of {len(proteins)} proteins have an OF3 model")
display(pd.Series(failed, name="error", dtype=str))

[1/151] A0A0B7GU52
  [skip] MSA for A0A0B7GU52 already exists
  [skip] OF3_A0A0B7GU52.cif already exists
[2/151] A0AAI7ZGD2
  [skip] MSA for A0AAI7ZGD2 already exists
  [skip] OF3_A0AAI7ZGD2.cif already exists
[3/151] B1MLW3
  [skip] MSA for B1MLW3 already exists
  [skip] OF3_B1MLW3.cif already exists
[4/151] E0SM38
  [skip] MSA for E0SM38 already exists
  [skip] OF3_E0SM38.cif already exists
[5/151] O14561
  [skip] MSA for O14561 already exists
  [skip] OF3_O14561.cif already exists
[6/151] O14625
  [skip] MSA for O14625 already exists
  [skip] OF3_O14625.cif already exists
[7/151] O14933
  [skip] MSA for O14933 already exists
  [skip] OF3_O14933.cif already exists
[8/151] O15392
  [skip] MSA for O15392 already exists
  [skip] OF3_O15392.cif already exists
[9/151] O43678
  [skip] MSA for O43678 already exists
  [skip] OF3_O43678.cif already exists
[10/151] O64980
  [skip] MSA for O64980 already exists
  [skip] OF3_O64980.cif already exists
[11/151] O95166
  [skip] MSA for O95166 alrea

Series([], Name: error, dtype: str)

In [24]:
# 4. Are the predictions valid? pLDDT, OF3 vs AF2
report = of3_report(proteins["uniprot_id"])
display(report[["pLDDT OF3", "pLDDT AF2"]].describe().round(1))
print(f"OF3 pLDDT >= 70: {(report['pLDDT OF3'] >= 70).sum()} of {len(report)}")
print(
    f"OF3 = UniProt sequence: {report['OF3 = UniProt sequence'].sum()} of {len(report)}"
)
display(report[~(report["pLDDT OF3"] >= 70) | ~report["OF3 = UniProt sequence"]])

,pLDDT OF3,pLDDT AF2
count,151.0,151.0
mean,73.3,87.6
std,10.7,8.9
min,38.9,39.7
25%,67.6,84.4
50%,73.5,89.8
75%,80.6,93.8
max,94.1,98.3


OF3 pLDDT >= 70: 101 of 151
OF3 = UniProt sequence: 151 of 151


,uniprot_id,OF3 = UniProt sequence,MSA depth,confidence,pLDDT OF3,pLDDT AF2
0,A0A0B7GU52,True,101 + 87,0.3,68.9,78.3
3,E0SM38,True,101 + 101,0.3,69.6,87.4
4,O14561,True,101 + 101,0.2,65.6,77.7
5,O14625,True,101 + 105,0.3,64.2,80.0
7,O15392,True,101 + 101,0.2,56.5,94.8
10,O95166,True,101 + 101,0.3,57.6,94.9
11,O95994,True,101 + 110,0.3,65.5,86.2
16,P01215,True,101 + 73,0.2,69.6,91.8
19,P02283,True,102 + 106,0.2,53.3,84.4
20,P02721,True,101 + 101,0.1,55.8,83.4


---
**Next:** notebook 01 checks that every chosen structure is on disk and readable, and describes what
the set is; notebook 02 runs the comparisons.